In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from keras_tuner.tuners import BayesianOptimization

from datetime import datetime
import pickle

## LOADING DATASET

In [2]:
# LOADING DATASET

PATH_TRAINING_A = '../data/preprocessing/normalized/normalized_local_original.npy'
lines_a_noisy = np.load(PATH_TRAINING_A)

train_norm, temp = train_test_split(lines_a_noisy, test_size=0.4, random_state=42)  # 40% for val+test
val_norm, test_norm = train_test_split(temp, test_size=0.25, random_state=42)   

print(f'Train Shape {train_norm.shape}, Val Shape {val_norm.shape}, Test Shape {test_norm.shape}')

Train Shape (41116, 64, 2), Val Shape (20559, 64, 2), Test Shape (6853, 64, 2)


## BUILDING AUTOENCODER ARCHITECTURE

In [ ]:
#AE ARCHITECTURE

timesteps = 64
features = 2
latent_dim = 32 

inputs = Input(shape=(timesteps, features))

# ENCODER (depth = 1, bidirectional)
x = Bidirectional(LSTM(128))(inputs)
encoded = Dense(latent_dim, activation='linear')(x)

# DECODER
x = RepeatVector(timesteps)(encoded)
x = LSTM(64, return_sequences=True)(x)
decoded = TimeDistributed(Dense(features))(x)

lstm_autoencoder = Model(inputs, decoded)
#lstm_autoencoder.summary()


## MODEL TRAINING

In [ ]:
#TRAINING SETUP 

description = 'mse_local_finetuned_bidrectional'
epochs = 50
batch_size = 32
loss = 'mse' #huber, mae
optimizer=tf.keras.optimizers.Adam(learning_rate=5.16e-3)

early_stop = EarlyStopping(
    monitor='val_loss',     # what to watch
    patience=20,            # epochs to wait for improvement
    restore_best_weights=True   
)

log_dir = "../checkpoints/autoencoder/logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# metrics: things to be observed, loss: what is actually used for learning.  ,
lstm_autoencoder.compile(optimizer=optimizer, loss=loss, metrics=['accuracy','mse', 'mae'])

In [ ]:
#TRAINING

timestamp = datetime.now().strftime('%d%m_%H%M')

history = lstm_autoencoder.fit(
    train_norm, train_norm,
    validation_data=(val_norm, val_norm),
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    callbacks=[early_stop, tensorboard_callback]
)


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir ../checkpoints/autoencoder/logs

In [8]:
# TRAINING HISTORY 

def plot_history(history, epochs, batch_size):
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Training Loss', color='#143642')

    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss', color='#EC9A29')

    plt.xlabel('Epochs', fontdict={'family': 'serif', 'size': 10})
    plt.ylabel('Loss', fontdict={'family': 'serif', 'size': 10})
    plt.legend(prop={'family': 'serif', 'size': 10})
    plt.title(f'Training Loss LSTM AE {epochs} Epochs', fontdict={'family': 'serif', 'size': 14})
    plt.xticks(fontsize=10, family='serif')
    plt.yticks(fontsize=10, family='serif')
    plt.grid()
    
    # save figure 
    save_path = f'../checkpoints/autoencoder/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_history(history, epochs, batch_size)

In [ ]:
# SAVING MODEL, WEIGHTS, HISTORY
lstm_autoencoder.save(f'../checkpoints/autoencoder/model/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.keras')
lstm_autoencoder.save_weights(f'../checkpoints/autoencoder/weights/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.weights.h5')
np.save(f'../checkpoints/autoencoder/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.npy', history.history)

## POST TRAINING

In [11]:
epochs=150
batch_size=32

In [ ]:
def plot_history(history, epochs, batch_size):
    plt.figure(figsize=(10, 5))
    plt.plot(history['loss'], label='Training Loss', color='#143642')

    if 'val_loss' in history:
        plt.plot(history['val_loss'], label='Validation Loss', color='#EC9A29')

    plt.xlabel('Epochs', fontdict={'family': 'serif', 'size': 10})
    plt.ylabel('Loss', fontdict={'family': 'serif', 'size': 10})
    plt.legend(prop={'family': 'serif', 'size': 10})
    plt.title(f'Training Loss LSTM AE {epochs} Epochs', fontdict={'family': 'serif', 'size': 14})
    plt.xticks(fontsize=10, family='serif')
    plt.yticks(fontsize=10, family='serif')
    plt.grid()
    
    # save figure 
    #save_path = f'../checkpoints/autoencoder/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.png'
    save_path = f'../checkpoints/autoencoder/history/bestModel_{batch_size}_batches_{epochs}_epochs.png'
    #plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

In [ ]:
# LOADING MODEL AND HISTORY

#lstm_autoencoder = tf.keras.models.load_model(f'../checkpoints/autoencoder/model/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.keras')
lstm_autoencoder = tf.keras.models.load_model(f'../checkpoints/autoencoder/model/2312_bestModel_32_batches_150_epochs.keras')
history = np.load(
    '../checkpoints/autoencoder/history/2312_bestModel_32_batches_150_epochs.npy',
    allow_pickle=True
).item()
#history
plot_history(history, epochs, batch_size)

In [13]:
reconstructed = lstm_autoencoder.predict(test_norm)

2026-01-05 16:20:04.587315: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step


In [14]:
# SAVING PREDICTION 

#with open(f'../data/results/lstm_autoencoder/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs_predictions.pkl', 'wb') as f:
with open(f'../data/results/lstm_autoencoder/2312_bestModel_{batch_size}_batches_{epochs}_epochs_predictions.pkl', 'wb') as f:
    pickle.dump(reconstructed, f)

In [ ]:
# PLOT PREDICTION
for i in range(1000,1036):
    plt.figure(figsize=(6, 10))

    plt.plot(test_norm[i,:,0], test_norm[i,:,1], color='#143642',  linestyle='--', label='Original')
    plt.plot(reconstructed[i,:,0], reconstructed[i,:,1], color='#EC9A29', label='Reconstructed')


    plt.xlabel('X', fontdict={'family': 'serif', 'size': 10})
    plt.ylabel('Y', fontdict={'family': 'serif', 'size': 10})

    plt.legend(prop={'family': 'serif', 'size': 10})
    plt.xticks(fontsize=10, family='serif')
    plt.yticks(fontsize=10, family='serif')
    plt.axis('equal')
    plt.grid(True)
    plt.title(f'LSTM AE Prediction - Line {i}', fontdict={'family': 'serif', 'size': 14})    
    
    # save figure 
    #save_path = f'../figures/predictions/prediction_line_{i}_{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.png'
    #plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()


## HYPERPARAMETER OPTIMIZATION

In [5]:
num_points = 64
coordinates = 2

def build_model(hp):
    # Latent dimension
    #latent_dim = hp.Choice('latent_dim', [8, 16, 32])
    
    # Depth (1,2,3 LSTM layers)
    depth = hp.Choice('depth', [1, 2, 3])
    
    # Hidden size options
    hidden_sizes = [8, 32, 64, 128]
    
    # Encoder LSTM units per layer
    enc_units = [hp.Choice(f'enc_lstm{i+1}', hidden_sizes) for i in range(depth)]
    
    # Decoder LSTM units per layer (mirror encoder)
    dec_units = [hp.Choice(f'dec_lstm{i+1}', hidden_sizes) for i in range(depth)]
    
    # Learning rate
    lr = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
    
    inputs = Input(shape=(num_points, coordinates))
    
    # ENCODER
    x = inputs
    for units in enc_units[:-1]:
        x = LSTM(units, return_sequences=True)(x)
    x = LSTM(enc_units[-1])(x)  # last layer without return_sequences
    
    #encoded = x
    # LATENT BOTTLE TBC
    latent = hp.Choice('latent_dim', [8, 16, 32])
    encoded = Dense(latent, activation='linear')(x)
    
    # DECODER
    x = RepeatVector(num_points)(encoded)
    for units in dec_units[:-1]:
        x = LSTM(units, return_sequences=True)(x)
    x = LSTM(dec_units[-1], return_sequences=True)(x)  # last layer returns sequences
    outputs = TimeDistributed(Dense(coordinates))(x)
    
    model = Model(inputs, outputs)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='mse',
        metrics=['mse', 'mae']
    )
    
    return model


In [6]:
tuner = BayesianOptimization(
    hypermodel=build_model,
    objective='val_loss',
    max_trials=20,          
    executions_per_trial=1,  
    directory= '../checkpoints/autoencoder/', 
    project_name='tuner_results'
)

Reloading Tuner from ../checkpoints/autoencoder/tuner_results/tuner0.json


In [ ]:
tuner.search(
    train_norm, train_norm,
    validation_data=(val_norm, val_norm),
    epochs=100,
    batch_size=32, #64
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]
)

Trial 20 Complete [04h 26m 25s]
val_loss: 3.3365311537636444e-05

Best val_loss So Far: 3.3365311537636444e-05
Total elapsed time: 3d 07h 22m 47s


In [8]:
best_hp = tuner.get_best_hyperparameters(1)[0]
best_model = tuner.get_best_models(1)[0]

print('Best hyperparameters:')
for p, v in best_hp.values.items():
    print(p, '=', v)


Best hyperparameters:
depth = 1
enc_lstm1 = 128
dec_lstm1 = 64
learning_rate = 0.005155665082994383
latent_dim = 32
enc_lstm2 = 64
dec_lstm2 = 32
enc_lstm3 = 8
dec_lstm3 = 64


/opt/anaconda3/envs/tfgeo/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [11]:
history = best_model.fit(
    train_norm, train_norm,
    validation_data=(val_norm, val_norm),
    epochs=150,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)]
)


Epoch 1/150
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 85s 66ms/step - loss: 5.9647e-05 - mae: 0.0053 - mse: 5.9647e-05 - val_loss: 3.7131e-05 - val_mae: 0.0043 - val_mse: 3.7131e-05
Epoch 2/150
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 88s 68ms/step - loss: 5.2778e-05 - mae: 0.0051 - mse: 5.2778e-05 - val_loss: 1.4478e-04 - val_mae: 0.0084 - val_mse: 1.4478e-04
Epoch 3/150
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 85s 66ms/step - loss: 6.5921e-05 - mae: 0.0056 - mse: 6.5921e-05 - val_loss: 4.1683e-05 - val_mae: 0.0047 - val_mse: 4.1683e-05
Epoch 4/150
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 84s 65ms/step - loss: 6.9541e-05 - mae: 0.0054 - mse: 6.9541e-05 - val_loss: 4.5638e-05 - val_mae: 0.0049 - val_mse: 4.5638e-05
Epoch 5/150
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 83s 64ms/step - loss: 5.0262e-05 - mae: 0.0048 - mse: 5.0262e-05 - val_loss: 4.7630e-05 - val_mae: 0.0048 - val_mse: 4.7630e-05
Epoch 6/150
1285/1285 ━━━━━━━━━━━━━━━━━━━━ 85s 66ms/step - loss: 5.3551e-05 - mae: 0.0051 - mse: 5.3551e-05 - val_loss: 3.2878e-05 - val_mae: 0.0

In [13]:
best_model.save(f'../checkpoints/autoencoder/model/bestModel.keras')
best_model.save_weights(f'../checkpoints/autoencoder/weights/bestModel.weights.h5')
np.save(f'../checkpoints/autoencoder/history/bestModel.npy', history.history)

In [10]:
best_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        67,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 64, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 64, 2)          │           130 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 288,488 (1.10 MB)

 Trainable params: 96,162 (375.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 192,326 (751.28 KB)